In [3]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [4]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [15]:
import torch.nn as nn
import torch

class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias= False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))



    def forward(self,x):
            b, num_tokens, d_in = x.shape
            queries = self.W_query(x)
            keys =self.W_key(x)
            values =  self.W_value(x)

            #add a head dimension to the shape
            keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
            values = values.view(b, num_tokens, self.num_heads, self.head_dim)
            queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
            

            #transpose no. of tokens and head to get all matrix values in a certain head

            keys = keys.transpose(1,2)
            values = values.transpose(1,2)
            queries = queries.transpose(1,2)

            attn_scores = queries @ keys.transpose(2, 3)

            mask_simple = self.mask.bool()[:num_tokens, :num_tokens]

            attn_scores.masked_fill_(mask_simple, -torch.inf)
            attn_weights = torch.softmax(attn_scores/keys.shape[-1] ** 0.5, dim=-1)
            attn_weights = self.dropout(attn_weights)

            context_vec = (attn_weights @ values).transpose(1,2) #reformatted context vectors
            context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out) # concat the context vector

            context_vec = self.out_proj(context_vec) # optional projection for taking out importance from each attention head

            return context_vec


batch_size, context_length, d_in = batch.shape
d_out = 4

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, 2)


context_vecs = mha(batch)


context_vecs

            

        







tensor([[[ 0.4477, -0.0962,  0.1349,  0.0866],
         [ 0.3863, -0.0654,  0.1241,  0.1879],
         [ 0.3653, -0.0520,  0.1221,  0.2229],
         [ 0.3278,  0.0142,  0.1484,  0.2185],
         [ 0.3394,  0.0538,  0.1774,  0.2014],
         [ 0.3121,  0.0673,  0.1748,  0.2113]],

        [[ 0.4477, -0.0962,  0.1349,  0.0866],
         [ 0.3863, -0.0654,  0.1241,  0.1879],
         [ 0.3653, -0.0520,  0.1221,  0.2229],
         [ 0.3278,  0.0142,  0.1484,  0.2185],
         [ 0.3394,  0.0538,  0.1774,  0.2014],
         [ 0.3121,  0.0673,  0.1748,  0.2113]]], grad_fn=<UnsafeViewBackward0>)